# Mechanistic Interpretability Starter Notebook (Pythia-70M)

This notebook walks through:

1. Loading **Pythia-70M** with **TransformerLens**
2. Running simple forward passes and inspecting logits
3. Using **Residual Stream probing** (a.k.a. **Logit Lens**)
4. Doing **Activation Patching**
5. Visualizing attention patterns with **circuitsvis**


## Setup and Installation

This notebook explores **mechanistic interpretability (MI)** techniques using a small, fully-open transformer model.  
We rely on **TransformerLens**, which exposes internal activations, attention patterns, and residual streams in a way that makes causal analysis possible.

**Why Pythia-70M?**
- Small enough to run locally and iterate quickly
- Architecturally faithful to large GPT-style transformers
- Well-studied in the interpretability literature

The goal is not performance, but **mechanistic clarity** — understanding *how* specific outputs arise from internal computation.


In [1]:
import sys, subprocess

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', pkg], check=False)

pip_install('transformer-lens')
pip_install('circuitsvis')
pip_install('einops')
pip_install('torch')

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 9.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 11.1 MB/s  0:00:01 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 11.1 MB/s  0:00:00 11.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 10.6 MB/s  0:00:03 10.7 MB/s eta 0:00:01
Using cached markdown_it_py-

## Imports and Device Setup

In [3]:
import torch
from transformer_lens import HookedTransformer, utils
from transformer_lens.utils import get_act_name
import numpy as np
import matplotlib.pyplot as plt
import circuitsvis as cv
from typing import List, Dict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

## Model Loading

We load the Pythia-70M model using `HookedTransformer`, which wraps a standard transformer with:
- Named activation hooks
- Consistent tensor shapes across layers
- Easy access to attention heads and residual streams


In [4]:
model_name = 'pythia-70m-deduped'

model = HookedTransformer.from_pretrained(
    model_name,
    device=device,
    fold_ln=True,
    center_writing_weights=True,
    center_unembed=True,
)
model

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model pythia-70m-deduped into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()


## Tokenization and Forward Pass

Tokenization is performed with the model’s native tokenizer to ensure:
- Token boundaries align with training
- Logit-level interpretations are valid

At this stage, we are establishing the **forward computational graph** that all later interpretability techniques will interrogate.

Before intervening in the model, we run a clean forward pass on a prompt.

This establishes:
- Baseline logits
- Top-k predictions
- Reference activations for later causal comparisons

**Interpretability principle:**  
Always understand the model’s *natural behavior* before modifying it.  
All patching, probing, and attribution is relative to this baseline.

In [5]:
def to_tokens(text):
    return model.to_tokens(text, prepend_bos=True).to(device)

def to_str(tokens):
    return model.to_string(tokens)

def get_top_predictions(logits, top_k=5):
    last_logits = logits[0, -1, :]
    probs = last_logits.softmax(dim=-1)
    top_probs, top_indices = probs.topk(top_k)
    tokens = [model.to_string(idx.unsqueeze(0)) for idx in top_indices]
    return list(zip(tokens, top_probs.detach().cpu().numpy()))

prompt = 'The capital of France is'
tokens = to_tokens(prompt)
logits = model(tokens)
get_top_predictions(logits)

[(' the', 0.14661531),
 (' a', 0.08236917),
 (' not', 0.040534925),
 (' in', 0.03654322),
 (' to', 0.027328452)]

## Residual Stream Probing (Logit Lens)

The **residual stream** is the central communication channel of a transformer.  
At each layer, attention and MLP outputs are *added* to this stream.

The **Logit Lens** projects intermediate residual states directly into vocabulary space using the unembedding matrix.

This allows us to ask:
> *“If the model stopped computing at this layer, what would it predict?”*

**What to look for:**
- Early layers: vague or incorrect predictions
- Middle layers: partial semantic structure
- Late layers: sharp convergence on the final answer

This reveals *where* information becomes linearly decodable and how representations sharpen over depth.


In [8]:
import torch
from transformer_lens.utils import get_act_name

def logit_lens(
    text: str,
    token_pos: int = -1,
    max_layers: int | None = 6,
    top_k: int = 5,
    prefer: tuple[str, ...] = ("resid_mid", "resid_post", "resid_pre"),
):
    tokens = to_tokens(text)
    logits, cache = model.run_with_cache(tokens)

    # Resolve negative token positions
    if token_pos < 0:
        token_pos = tokens.shape[1] + token_pos

    # Which layers exist?
    n_layers = model.cfg.n_layers
    if max_layers is None:
        max_layers = n_layers
    max_layers = min(max_layers, n_layers)

    # Helper: pick a working residual hook name for a given layer
    def first_available_resid(layer: int) -> str:
        for name in prefer:
            key = get_act_name(name, layer)  # e.g. blocks.0.hook_resid_post
            if key in cache.cache_dict:
                return key
        # Fallback: dump a helpful hint if nothing matched
        # (this should be rare)
        available = [k for k in cache.cache_dict.keys() if f"blocks.{layer}." in k and "resid" in k]
        raise KeyError(
            f"No residual hook found for layer {layer}. Tried {prefer}. "
            f"Available resid keys for that layer: {available[:20]}"
        )

    W_U = model.W_U  # [d_model, d_vocab]

    for layer in range(max_layers):
        resid_key = first_available_resid(layer)
        resid = cache[resid_key][0, token_pos, :]          # [d_model]
        pseudo_logits = resid @ W_U                        # [d_vocab]
        probs = torch.softmax(pseudo_logits, dim=-1)

        top_probs, top_indices = torch.topk(probs, k=top_k)

        decoded = [model.to_string(idx.unsqueeze(0)) for idx in top_indices]
        pairs = list(zip(decoded, top_probs.tolist()))
        print(f"Layer {layer} ({resid_key.split('.')[-1]}): {pairs}")

# Example
logit_lens("The capital of France is", max_layers=6, top_k=5)


Layer 0 (hook_resid_post): [(' finding', 0.11787395179271698), (' true', 0.01868196576833725), (' supposed', 0.01707020215690136), (' hereditary', 0.012869569472968578), (' nearly', 0.011054741218686104)]
Layer 1 (hook_resid_post): [(' thick', 0.04069048911333084), (' stacking', 0.03769131004810333), (' slow', 0.03436145931482315), (' purple', 0.022750146687030792), ('hes', 0.021381936967372894)]
Layer 2 (hook_resid_post): [(' White', 0.11652210354804993), (' Treasury', 0.05161827802658081), (' red', 0.0494736023247242), (' Purple', 0.03533800691366196), (' still', 0.028652409091591835)]
Layer 3 (hook_resid_post): [(' alive', 0.16198374330997467), (' highly', 0.07134000211954117), (' also', 0.052716419100761414), (' unchanged', 0.03689771518111229), (' welcomed', 0.03503391519188881)]
Layer 4 (hook_resid_post): [(' also', 0.3836720883846283), (' Luxem', 0.08181028068065643), (' Notre', 0.06873877346515656), (' Paris', 0.043749481439590454), (' dedicated', 0.04129748418927193)]
Layer 5 

## Activation Patching (Causal Tracing)

**Activation patching** is a causal intervention technique.

Procedure:
1. Run a **clean** forward pass
2. Run a **corrupted** forward pass (e.g., altered prompt)
3. Replace specific activations in the corrupted run with those from the clean run
4. Measure how much the original prediction is restored

This answers:
> *“Which internal components are causally responsible for this behavior?”*

Common patching targets:
- Residual stream at a specific layer
- Attention head outputs
- MLP activations

**Interpretation:**
- Large recovery ⇒ component is causally important
- No recovery ⇒ component is not necessary for this behavior


In [16]:
import torch
from transformer_lens.utils import get_act_name

def pick_resid_hook_for_layer(cache, layer: int, prefer=("resid_mid", "resid_post", "resid_pre")) -> str:
    for name in prefer:
        key = get_act_name(name, layer)
        if key in cache.cache_dict:
            return key
    available = [k for k in cache.cache_dict.keys() if f"blocks.{layer}." in k and "resid" in k]
    raise KeyError(f"No residual hook found for layer {layer}. Tried {prefer}. Available: {available[:30]}")

def find_token_pos(tokens, target_str: str) -> int:
    """
    Find the FIRST position i such that model.to_string(tokens[0,i]) == target_str.
    """
    for i in range(tokens.shape[1]):
        s = model.to_string(tokens[0, i].unsqueeze(0))
        if s == target_str:
            return i
    raise ValueError(f"Token {target_str!r} not found. Tokens were: {[model.to_string(tokens[0,i].unsqueeze(0)) for i in range(tokens.shape[1])] }")

def prob_of_token(logits: torch.Tensor, tok_id: int, pos: int = -1) -> float:
    probs = torch.softmax(logits[0, pos, :], dim=-1)
    return float(probs[tok_id].item())

def patch_and_score(
    layer: int,
    clean_cache,
    corrupted_tokens,
    hook_name: str,
    clean_pos: int,
    corr_pos: int,
    score_tok_id: int,
    score_pos: int = -1,
):
    def patch_hook(corrupted_act, hook):
        patched = corrupted_act.clone()
        patched[0, corr_pos, :] = clean_cache[hook_name][0, clean_pos, :]
        return patched

    patched_logits = model.run_with_hooks(
        corrupted_tokens,
        fwd_hooks=[(hook_name, patch_hook)],
    )
    return prob_of_token(patched_logits, score_tok_id, pos=score_pos)

# Prompts
clean_prompt = "Alice loves"
corrupted_prompt = "Bob loves"

clean_tokens = to_tokens(clean_prompt)
corrupted_tokens = to_tokens(corrupted_prompt)

_, clean_cache = model.run_with_cache(clean_tokens)

# Find the shared anchor token position: ' loves'
clean_pos = find_token_pos(clean_tokens, " loves")
corr_pos  = find_token_pos(corrupted_tokens, " loves")

# What do we want the next token to be?
# NOTE: leading space matters for GPT-style tokenization.
bob_id   = model.to_single_token(" Bob")
alice_id = model.to_single_token(" Alice")

# Baselines
base_corr_logits = model(corrupted_tokens)
print("Baseline corrupted P(' Bob'):", prob_of_token(base_corr_logits, bob_id))
print("Baseline corrupted P(' Alice'):", prob_of_token(base_corr_logits, alice_id))

# Patch across layers
for layer in range(model.cfg.n_layers):
    hook_name = pick_resid_hook_for_layer(clean_cache, layer, prefer=("resid_mid", "resid_post", "resid_pre"))
    p_bob = patch_and_score(
        layer=layer,
        clean_cache=clean_cache,
        corrupted_tokens=corrupted_tokens,
        hook_name=hook_name,
        clean_pos=clean_pos,
        corr_pos=corr_pos,
        score_tok_id=bob_id,
        score_pos=-1,
    )
    print(f"Layer {layer} ({hook_name.split('.')[-1]}): patched P(' Bob') = {p_bob:.6f}")



Baseline corrupted P(' Bob'): 0.0006067963549867272
Baseline corrupted P(' Alice'): 0.001994148828089237
Layer 0 (hook_resid_post): patched P(' Bob') = 0.000538
Layer 1 (hook_resid_post): patched P(' Bob') = 0.000533
Layer 2 (hook_resid_post): patched P(' Bob') = 0.000359
Layer 3 (hook_resid_post): patched P(' Bob') = 0.000365
Layer 4 (hook_resid_post): patched P(' Bob') = 0.000117
Layer 5 (hook_resid_post): patched P(' Bob') = 0.000110


In [14]:
def show_tokens(text: str):
    toks = to_tokens(text)[0]
    for i, t in enumerate(toks):
        print(i, repr(model.to_string(t.unsqueeze(0))))
    return toks

show_tokens(clean_prompt)
show_tokens(corrupted_prompt)


0 '<|endoftext|>'
1 'Al'
2 'ice'
3 ' loves'
0 '<|endoftext|>'
1 'Bob'
2 ' loves'


tensor([    0, 26845, 14528])

## Attention Pattern Visualization

Attention maps show **which tokens attend to which other tokens** at each head and layer.

These patterns often encode:
- Induction (copying previous tokens)
- Syntax (subject–verb relationships)
- Positional or delimiter tracking

However, attention ≠ importance.

**Key caution:**  
A head can attend strongly without being causally relevant.  
Attention visualizations are best used *in conjunction* with activation patching.

Use attention maps to generate hypotheses — then test them causally.


In [18]:
from transformer_lens.utils import get_act_name

test_prompt = "Alice loves Bob and Bob loves Alice because"
tokens = to_tokens(test_prompt)
_, cache = model.run_with_cache(tokens)

layer = 0
layer_keys = sorted([k for k in cache.cache_dict.keys() if k.startswith(f"blocks.{layer}.")])

# Print anything attention-related
for k in layer_keys:
    if "attn" in k or "pattern" in k:
        print(k)

blocks.0.attn.hook_attn_scores
blocks.0.attn.hook_k
blocks.0.attn.hook_pattern
blocks.0.attn.hook_q
blocks.0.attn.hook_rot_k
blocks.0.attn.hook_rot_q
blocks.0.attn.hook_v
blocks.0.attn.hook_z
blocks.0.hook_attn_out


In [20]:
import numpy as np

test_prompt = "Alice loves Bob and Bob loves Alice because"
tokens = to_tokens(test_prompt)
_, cache = model.run_with_cache(tokens)

layer_to_viz = 0

# Newer TransformerLens key
pattern_key = f"blocks.{layer_to_viz}.attn.hook_pattern"

attn_pattern = cache[pattern_key][0].detach().cpu().numpy()  # [n_heads, seq, seq]

# Make a per-token string list (safe across versions)
tok_strs = [model.to_string(t.unsqueeze(0)) for t in tokens[0]]

cv.attention.attention_patterns(attention=attn_pattern, tokens=tok_strs)


In [21]:
head_to_viz = 0
cv.attention.attention_patterns(
    attention=attn_pattern[head_to_viz:head_to_viz+1],
    tokens=tok_strs
)


- Rows = query tokens (“where attention is coming from”)

- Columns = key tokens (“where attention is going to”)

- Color intensity = how strongly one token attends to another

- Each small square labeled Head 0 … Head 7 is one attention head

- All heads shown are from the same transformer layer

## Mechanistic Interpretability Workflow

A practical MI workflow looks like this:

1. **Baseline forward pass** — understand the model’s natural output
2. **Logit Lens** — locate where information emerges
3. **Activation patching** — test causal necessity
4. **Attention inspection** — identify candidate circuits
5. **Targeted re-patching** — isolate minimal mechanisms

This mirrors experimental science:
- Observation → hypothesis → intervention → validation

Small models allow this loop to run quickly and transparently.


## Sparse Autoencoders (SAEs)

Sparse Autoencoders are used to **decompose dense activations into sparse, interpretable features**.

Instead of asking:
> “What does this neuron do?”

We ask:
> “What *features* does this activation space represent?”

SAEs are trained to:
- Reconstruct activations accurately
- Use as few active features as possible (sparsity)

In MI, SAEs help:
- Discover latent concepts
- Identify reusable computational features
- Move beyond neuron-level analysis

They are especially powerful when applied to the **residual stream**.


## Limitations and Best Practices

Mechanistic interpretability is powerful but fragile.

Best practices:
- Use multiple prompts
- Control for distribution shift when patching
- Avoid over-interpreting single examples
- Validate findings across layers and seeds

MI reveals *mechanisms*, not intent — careful experimental design matters.
